# Messages 与 Runnable 协议

对标文档：[LangChain > Core Components > Messages](https://python.langchain.com/docs/concepts/messages/)

In [1]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_learning.llm import get_llm

llm = get_llm()

**术语：**
- **Message（消息）** = LangChain 中所有组件间传递的数据单元
- **Runnable（可运行组件）** = 实现了统一接口的组件，可以使用 `invoke`、`stream`、`batch` 调用

```mermaid
%%{init: {'theme': 'dark'}}%%
graph LR
    A[消息类型] --> B[SystemMessage 系统消息]
    A --> C[HumanMessage 用户消息]
    A --> D[AIMessage AI 回复]
    A --> E[ToolMessage 工具结果]
    F[Runnable 协议] --> G[invoke 单次调用]
    F --> H[stream 流式输出]
    F --> I[batch 批量处理]
```

### 1. 消息类型

LangChain 用不同消息类型区分对话中的角色，每种消息携带不同的元数据。

In [2]:
system_msg = SystemMessage("你是编程助手")
human_msg = HumanMessage("Python 和 Java 的区别？")

messages = [system_msg, human_msg]
response: AIMessage = llm.invoke(messages)

print(f"消息类型: {type(response).__name__}")
print(f"内容: {response.content[:100]}...")
print(f"元数据: {list(response.response_metadata.keys())}")

消息类型: AIMessage
内容: Python 和 Java 是两种非常流行的编程语言，它们各有特点，适用于不同的场景。以下是它们的主要区别：

### 1. **语法风格**
- **Python**：语法简洁、易读，使用缩进来定义...
元数据: ['token_usage', 'model_name', 'system_fingerprint', 'id', 'service_tier', 'finish_reason', 'logprobs']


**术语：**
- **SystemMessage** = 系统消息，设定 AI 的角色和行为
- **HumanMessage** = 用户消息，代表用户的输入
- **AIMessage** = AI 的回复，包含 `content`（文本）和 `response_metadata`（模型返回的元数据如 token 用量）
- **ToolMessage** = 工具调用返回的结果消息

所有消息类型都有 `content`（文本内容）和 `type`（消息类型字符串，如 `"human"`、`"ai"`）。

### 2. AIMessage 与工具调用

当模型决定调用工具时，AIMessage 会包含 `tool_calls` 字段。

In [3]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """两数相加"""
    return a + b

llm_with_tools = llm.bind_tools([add])
response = llm_with_tools.invoke("3 加 5 等于几？")

if response.tool_calls:
    print(f"工具名称: {response.tool_calls[0]['name']}")
    print(f"参数: {response.tool_calls[0]['args']}")

工具名称: add
参数: {'a': 3, 'b': 5}


**术语：**
- **tool_calls** = AIMessage 上的属性，记录模型想调用的工具及其参数
- **bind_tools** = 把工具注册到 LLM 上，让模型知道可以使用哪些工具

每个 `tool_calls` 元素包含：`name`（工具名）、`args`（参数字典）、`id`（调用 ID）。后续可以通过 ToolMessage 把工具结果返回给模型。

### 3. Runnable 协议

LangChain 中的所有组件（模型、提示词模板、解析器）都实现了 Runnable 接口，提供统一的调用方式。

In [4]:
chain = ChatPromptTemplate.from_template("用一句话解释{concept}") | llm | StrOutputParser()

# invoke — 单次调用
result = chain.invoke({"concept": "Runnable 协议"})
print(f"invoke: {result[:60]}...\n")

# stream — 流式输出
print("stream: ", end="")
for chunk in chain.stream({"concept": "消息传递"}):
    print(chunk, end="", flush=True)
print("\n")

# batch — 批量处理
results = chain.batch([
    {"concept": "Python"},
    {"concept": "Rust"},
    {"concept": "Go"},
])
for i, r in enumerate(results):
    print(f"batch[{i}]: {r[:40]}...")

invoke: Runnable 协议是一个定义了一组必须实现的方法（如 `run`）的接口，用于让对象在独立线程中执行任务，从而实现多...

stream: 消息传递是一种通过发送和接收数据包（消息）在独立实体（如进程、服务或系统）之间进行通信的机制，它允许它们在不共享内存的情况下交换信息、协调动作或触发事件。

batch[0]: Python是一种高级、解释型、面向对象的编程语言，以其简洁易读的语法和强大的标...
batch[1]: Rust 是一种系统编程语言，它通过所有权、借用和生命周期机制在编译时保证内存安...
batch[2]: Go是一种由Google开发的静态类型、编译型编程语言，旨在通过简洁的语法、高效...


**术语：**
- **invoke(input)** = 单次调用，返回完整结果
- **stream(input)** = 流式调用，逐个 yield 数据块
- **batch(inputs)** = 批量处理，传入列表，返回列表

```mermaid
%%{init: {'theme': 'dark'}}%%
graph LR
    A[Runnable 接口] --> B[invoke 单个入单个出]
    A --> C[stream 单个入逐个出]
    A --> D[batch 多个入多个出]
```

### 4. Runnable 组合与绑定

除了 `|` 管道组合，Runnable 还提供了 `bind`、`with_config`、`RunnablePassthrough` 和 `RunnableParallel` 等高级用法。

In [5]:
from langchain_core.runnables import RunnableConfig

# bind — 绑定固定参数
bind_chain = ChatPromptTemplate.from_template("写一句关于{topic}的诗") | llm.bind(stop=["."])
print(f"bind(stop): {bind_chain.invoke({'topic': '春天'}).content}...\n")

# RunnablePassthrough — 透传，配合字典构建输入
passthrough_chain = (
    {"topic": RunnablePassthrough()}
    | ChatPromptTemplate.from_template("关于{topic}，一句话")
    | llm | StrOutputParser()
)
print(f"RunnablePassthrough: {passthrough_chain.invoke('编程')[:60]}...\n")

# RunnableParallel — 并行执行多条链
c1 = ChatPromptTemplate.from_template("用一句话总结{concept}") | llm | StrOutputParser()
c2 = ChatPromptTemplate.from_template("列出{concept}的两个缺点") | llm | StrOutputParser()
parallel = RunnableParallel(summary=c1, cons=c2)
result = parallel.invoke({"concept": "微服务架构"})
print(f"Parallel[summary]: {result['summary'][:50]}...")
print(f"Parallel[cons]: {result['cons'][:50]}...")

bind(stop): 春水初生，春林初盛，春风十里不如你。...

RunnablePassthrough: 编程的本质，是用逻辑和抽象思维，将人类意图转化为机器可执行的精确指令。...

Parallel[summary]: 微服务架构是一种将单一应用程序划分为一组小型、独立部署的服务，每个服务围绕特定业务能力构建，并通过轻...
Parallel[cons]: 微服务架构虽然有很多优点，但也存在明显的缺点。以下是两个核心缺点：

1.  **系统复杂性显著增加...


**术语：**
- **bind(kwargs)** = 给 Runnable 绑定额外参数，如 `stop`、`response_format`，不修改原对象
- **RunnablePassthrough()** = 透传组件，输入什么就输出什么，常用于构建字典输入
- **RunnableParallel** = 并行执行多个 Runnable，各自输出到不同 key

```mermaid
%%{init: {'theme': 'dark'}}%%
graph LR
    A[Runnable 组合] --> B[bind 绑定参数]
    A --> C[RunnablePassthrough 透传]
    A --> D[RunnableParallel 并行]
    D --> E[chain1 → key1]
    D --> F[chain2 → key2]
```